## Project 2: Content-Based and Collaborative Filtering

**Course:** DATA 612 – Recommender Systems<br>
**Student:** Inna Yedzinovich

### Phase 0 — Data Acquisition and Preparation

For this project, we used a MovieLens-based dataset available at:
https://grouplens.org/datasets/movielens/ml_belief_2024/
The dataset includes:

 - user_rating_history.csv: user–movie ratings
 - movies.csv: movie metadata (titles and genres)

The ratings dataset provides the core structure required for building a recommender system, in the form:
| userId | movieId | rating |


The full dataset is large, so we created a smaller subset of approximately 50 users and 50 movies.

This was done to:

 - reduce computational complexity
 - make the system easier to debug and interpret
 - maintain sufficient data density for similarity calculations

Instead of randomly sampling data, we selected:

 - users with a high number of ratings
 - movies that were rated frequently

This ensures the resulting user–item matrix is not too sparse.

Data preparation was performed using Python (pandas), with assistance from Copilot.


In [1]:
# imports
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
ratings = pd.read_csv("data/user_rating_history.csv")
movies = pd.read_csv("data/movies.csv")

# Select users with many ratings
user_counts = ratings['userId'].value_counts()
active_users = user_counts[user_counts >= 50].index[:50]

subset = ratings[ratings['userId'].isin(active_users)]

# Select frequently rated movies
movie_counts = subset['movieId'].value_counts()
popular_movies = movie_counts[movie_counts >= 50].index[:50]

subset = subset[subset['movieId'].isin(popular_movies)]

# Fix duplicate issue with pivot_table
matrix = subset.pivot_table(
    index='userId',
    columns='movieId',
    values='rating',
    aggfunc='mean'
)

print("Matrix shape:", matrix.shape)
print(matrix.head())

Matrix shape: (50, 50)
movieId  111     260     356     541       924     1196    1197    1198    \
userId                                                                      
55083       NaN     4.5    4.75     NaN  3.833333     3.5     4.0    3.25   
102175      4.5     4.0    4.50     4.5  2.000000     4.0     4.0    4.00   
114150      5.0     5.0    4.75     3.0  2.000000     5.0     5.0    5.00   
121987      2.5     3.0    2.50     3.0  3.000000     4.0     4.0    3.00   
125273      5.0     5.0    5.00     5.0  5.000000     5.0     5.0    5.00   

movieId  1200    1206    ...  168250  170705  177615  179401  182715  187593  \
userId                   ...                                                   
55083       NaN     NaN  ...     NaN    4.25     NaN     NaN     NaN     NaN   
102175      4.0     4.5  ...     NaN    4.50     3.5     2.0     NaN     NaN   
114150      5.0     4.0  ...     4.0    4.50     NaN     3.5     1.5     4.5   
121987      3.0     4.0  ...     5.0 

### Phase 1 — User–User Collaborative Filtering

In this phase, we implement User–User Collaborative Filtering.

The idea is:
 - Users with similar rating behavior will have similar preferences.
 - We use this similarity to predict missing ratings.

First we need to handle missing values before computing similarity.
Replacing NaN with 0 means that “unknown rating” is treated as no interaction, and vector lengths is consistent,

In [3]:
matrix_filled = matrix.fillna(0)

Second, we compute cosine similarity between all user vectors.

In [4]:
user_similarity = cosine_similarity(matrix_filled)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=matrix.index,
    columns=matrix.index
)

print(user_similarity_df.head())

userId    55083     102175    114150    121987    125273    130215    132325  \
userId                                                                         
55083   1.000000  0.715479  0.695231  0.659372  0.730285  0.567512  0.537862   
102175  0.715479  1.000000  0.876596  0.870347  0.875983  0.864849  0.664731   
114150  0.695231  0.876596  1.000000  0.888255  0.849373  0.894796  0.628590   
121987  0.659372  0.870347  0.888255  1.000000  0.853054  0.912907  0.661328   
125273  0.730285  0.875983  0.849373  0.853054  1.000000  0.823039  0.657166   

userId    149153    152177    158981  ...    356999    357819    361004  \
userId                                ...                                 
55083   0.630663  0.691754  0.431085  ...  0.626831  0.673736  0.479436   
102175  0.589370  0.900263  0.604959  ...  0.633523  0.897243  0.648113   
114150  0.669012  0.916457  0.580517  ...  0.579984  0.849379  0.597348   
121987  0.583637  0.922954  0.555571  ...  0.649389  0.915144  0

Third, we convert to labeled DataFrame and predict ratings using weighted average of other users’ ratings using the followong formula:
- prediction = (ratings × similarity) / sum(similarity)

What this does:
 - .dot: performs matrix multiplication
 - each cell becomes a weighted combination of similar users
 - denominator normalizes influence

In [5]:
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=matrix.index,
    columns=matrix.index
)

print(user_similarity_df.head())

predicted_ratings = user_similarity_df.dot(matrix_filled) / user_similarity_df.sum(axis=1).values.reshape(-1, 1)
print(predicted_ratings.head())


userId    55083     102175    114150    121987    125273    130215    132325  \
userId                                                                         
55083   1.000000  0.715479  0.695231  0.659372  0.730285  0.567512  0.537862   
102175  0.715479  1.000000  0.876596  0.870347  0.875983  0.864849  0.664731   
114150  0.695231  0.876596  1.000000  0.888255  0.849373  0.894796  0.628590   
121987  0.659372  0.870347  0.888255  1.000000  0.853054  0.912907  0.661328   
125273  0.730285  0.875983  0.849373  0.853054  1.000000  0.823039  0.657166   

userId    149153    152177    158981  ...    356999    357819    361004  \
userId                                ...                                 
55083   0.630663  0.691754  0.431085  ...  0.626831  0.673736  0.479436   
102175  0.589370  0.900263  0.604959  ...  0.633523  0.897243  0.648113   
114150  0.669012  0.916457  0.580517  ...  0.579984  0.849379  0.597348   
121987  0.583637  0.922954  0.555571  ...  0.649389  0.915144  0

Next, we evaluate with RMSE (as we did in project 1 but now on a bigger dataset).

In [6]:
mask = ~matrix.isna()

# flatten and remove NaNs properly
y_true = matrix.values[mask.values]
y_pred = predicted_ratings.values[mask.values]

mse = mean_squared_error(y_true, y_pred)
rmse_user = np.sqrt(mse)

print("User-User RMSE:", rmse_user)

User-User RMSE: 1.5139936777170042


In this phase, we built a User–User Collaborative Filtering model using the user-item matrix from Phase 0. We calculated similarity between users using cosine similarity, where values closer to 1 mean users are more similar. Many similarity scores were fairly high because we selected active users who rate similar popular movies. Then, we predicted missing ratings by taking a weighted average of ratings from similar users. This was done using matrix multiplication, where each prediction is based on how similar other users are. Most predicted ratings ended up between 2 and 4, which is normal because averaging smooths out extreme values. Finally, we evaluated the model using RMSE, comparing predicted ratings with actual ratings where available.

### Phase 2 — Content-Based Filtering


In this phase, we recommend movies based on movie content, not other users. Instead of asking: "what do similar users like?", we ask: "what movies are similar to the ones the user already likes?"

What we are doing:
 - Clean genre text formatting
 - Convert text into feature vectors
 - Represent movies as numeric genre data
 - Compute movie-to-movie similarity
 - Measure similarity using cosine distance
 - Store similarities in labeled matrix

In [ ]:
movies_small = movies[movies['movieId'].isin(matrix.columns)].copy()
movies_small['genres_clean'] = movies_small['genres'].str.replace('|', ' ', regex=False)
print("Cleaned genres:")
print(movies_small[['movieId', 'genres_clean']].head())

vectorizer = CountVectorizer()
genre_matrix = vectorizer.fit_transform(movies_small['genres_clean'])
print("\nGenre matrix shape:", genre_matrix.shape)

movie_similarity = cosine_similarity(genre_matrix)
print("\nMovie similarity (numpy array preview):")
print(movie_similarity[:5, :5])

# rebuild SMALL similarity matrix
movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=movies_small['movieId'],
    columns=movies_small['movieId']
)

print("\nMovie similarity DataFrame:")
print(movie_similarity_df.head())

Cleaned genres:
   movieId                                 genres_clean
0        1  Adventure Animation Children Comedy Fantasy
1        2                   Adventure Children Fantasy
2        3                               Comedy Romance
3        4                         Comedy Drama Romance
4        5                                       Comedy

Genre matrix shape: (90638, 24)


Next, we are going combine the following matrices to make recommendations. :
 - user ratings (matrix)
 - movie similarity (movie_similarity_df)

Note: The kernel crash occurred because movie similarity was computed on the full dataset (~90,000 movies), resulting in a very large matrix (~90k × 90k), which exceeds available memory. Even slicing this matrix still requires it to be stored in memory. To fix this, similarity was recomputed only on the subset of movies used in the user-item matrix, reducing the size and preventing crashes.

In [ ]:
common_movies = matrix.columns.intersection(movie_similarity_df.index)
print("Number of common movies:", len(common_movies))

matrix_cb = matrix[common_movies]
movie_sim_cb = movie_similarity_df.loc[common_movies, common_movies]

print("\nFiltered user-item matrix:")
print(matrix_cb.head())

print("\nFiltered movie similarity matrix:")
print(movie_sim_cb.head())

matrix_cb_filled = matrix_cb.fillna(0)

predicted_cb = matrix_cb_filled.dot(movie_sim_cb) / movie_sim_cb.sum(axis=1)

print("\nPredicted ratings (content-based):")
print(predicted_cb.head())

In [ ]:
# Evaluate with RMSE
mask = ~matrix_cb.isna()

y_true = matrix_cb.values[mask.values]
y_pred = predicted_cb.values[mask.values]

mse = mean_squared_error(y_true, y_pred)
rmse_cb = np.sqrt(mse)

print("\nContent-Based RMSE:", rmse_cb)

In this phase, we implemented Content-Based Filtering using movie genre information. First, we converted the genres into a numerical format by transforming text into vectors, allowing each movie to be represented based on its genre features. Next, we computed cosine similarity between movies to measure how similar they are based on their genres, resulting in a movie-to-movie similarity matrix where values range between 0 and 1. Then, we predicted missing ratings by taking a weighted average of ratings for similar movies that a user has already rated. This was implemented using matrix multiplication, where each prediction is based on the similarity between movies and the user’s existing ratings. As a result, predicted ratings tend to fall within a moderate range, as the averaging process smooths extreme values. Finally, we evaluated the model using RMSE by comparing predicted ratings with actual ratings where available.

## Conclusion

Overall, both User–User Collaborative Filtering and Content-Based Filtering performed similarly, with RMSE around 1.5 and 1.4. The content-based model was slightly better, but the difference is small. This is expected because both methods are simple and use limited information. Collaborative filtering uses user rating behavior, but it is affected by sparse data and heavy averaging across users. Content-based filtering uses only genre features, which is a very basic representation and does not capture deeper preferences like actors, story, or style. In both cases, predictions are calculated using weighted averages, which smooth out ratings and pull them toward the middle, making it harder to capture very high or very low ratings. Because of these limitations, both models end up with similar performance and work mainly as baseline approaches rather than highly accurate recommendation systems.